# OCR Processor

Mistral Document AI API comes with a Document OCR (Optical Character Recognition) processor, powered by our latest OCR model `mistral-ocr-latest`, which enables you to extract text and structured content from PDF documents.

This notebook demonstrates how to upload a PDF file to Mistral Cloud and get the OCR results from the uploaded PDF by retrieving a signed url.


In [1]:
import getpass

MISTRAL_API_KEY = getpass.getpass("Enter your API KEY: ")

Enter your API KEY:  ········


In [2]:
from mistralai import Mistral

client = Mistral(api_key=MISTRAL_API_KEY)

In [3]:
# Upload a file to Mistral Cloud

from pathlib import Path

path = Path(input())

uploaded_pdf = client.files.upload(
    file={
        "file_name": path.name,
        "content": open(path, "rb"),
    },
    purpose="ocr"
)  

 data/LoRA.pdf


In [4]:
# Get a signed url to access the file

signed_url = client.files.get_signed_url(file_id=uploaded_pdf.id)

In [5]:
# Query the OCR endpoint with the signed url

ocr_response = client.ocr.process(
    model="mistral-ocr-latest",
    document={
        "type": "document_url",
        "document_url": signed_url.url,
    },
    table_format="html", # default is None
    #extract_header=True, # default is False
    #extract_footer=True, # default is False
    #include_image_base64=True
)

In [15]:
# Extract the pages as one .md file

document = ""
for page in ocr_response.pages:
    document+=page.markdown

# Preview
print(document[:300]+"...\n")
print(*[" "*50+"."]*3, sep="\n")
print("\n..."+document[-300:])

# LORA: LOW-RANK ADAPTATION OF LARGE LANGUAGE MODELS

Edward Hu* Yelong Shen* Phillip Wallis Zeyuan Allen-Zhu

Yuanzhi Li Shean Wang Lu Wang Weizhu Chen

Microsoft Corporation

{edwardhu, yeshe, phwallis, zeyuana,

yuanzhil, swang, luw, wzchen}@microsoft.com

yuanzhil@andrew.cmu.edu

(Version 2)

# ...

                                                  .
                                                  .
                                                  .

...n the singular directions of  $W_{q}$  and those of  $\Delta W_{q}$  with varying  $r$  and a random baseline.  $\Delta W_{q}$  amplifies directions that are important but not emphasized in  $W$ .  $\Delta W$  with a larger  $r$  tends to pick up more directions that are already emphasized in  $W$ .


In [7]:
# Export to markdown

def export_to_md(content: str, filename: str = "output.md"):
    """Export a string as a Markdown file."""
    if not filename.endswith(".md"):
        filename += ".md"
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    
    print(f"Exported to {filename}")

filename = str(path.parent / path.stem)
export_to_md(document, filename)

Exported to data/LoRA.md


In [8]:
# Delete the pdf file from Mistral cloud unless you wish to reuse it later:

client.files.delete(file_id=uploaded_pdf.id)

DeleteFileOut(id='e385d2ce-21fc-4000-b7ac-5d941586f299', object='file', deleted=True)

---